In [1]:
import requests
from pathlib import Path

# GitHub repository directory
api_url = "https://api.github.com/repos/zyxl100/foraging_MQLP_2026/contents/run_exp/data" # specifying the github data url

# Local destination
save_dir = Path(
    r"C:\Users\zyfxl\OneDrive - UC Irvine\UCI\CCNL Lab\Foraging Task Overview\foraging_MQLP-main\foraging_MQLP-main\replication sample\raw data\batch3" # specifying local url
)

save_dir.mkdir(parents=True, exist_ok=True)

# Get list of files from GitHub
response = requests.get(api_url, params={"ref": "main"}, timeout=30)
response.raise_for_status()

# Download all CSV files
for file in response.json():
    if file["type"] == "file" and file["name"].lower().endswith(".csv"):
        filename = file["name"]
        file_response = requests.get(file["download_url"], timeout=120)
        file_response.raise_for_status()

        (save_dir / filename).write_bytes(file_response.content)
        print(f"Downloaded: {filename}")

print("All CSV files downloaded successfully!")

Downloaded: S_56e32190e414dc000c8b40ae.csv
Downloaded: S_5d23c74c557d52001c823711.csv
Downloaded: S_5d4a41890e604c00011ade8b.csv
Downloaded: S_5e379ad54dd79670c8572f86.csv
Downloaded: S_6078b4ad9006e4f6572dde97.csv
Downloaded: S_6329e23d509947bfd4c50ab3.csv
Downloaded: S_65bf9aeb592e888df327d648.csv
Downloaded: S_671bbaf1bec15d847b871b9b.csv
Downloaded: S_672c6d81f10d775112a46f36.csv
Downloaded: S_697a403de832304dc7b0af9e.csv
Downloaded: S_698e0458100300aaa3b5302e.csv
Downloaded: S_698e819429f8b9faad03c077.csv
Downloaded: S_699293129f0baf693f34d7db.csv
Downloaded: S_6992e04a1733a51d3579b194.csv
Downloaded: S_69b7603c614bdbf78dd9517d.csv
Downloaded: S_69c635309ff3a6290637e557.csv
Downloaded: S_69f132c7c750d3abc3c7bf4a.csv
Downloaded: S_6a04cfd5f238e290a0746a15.csv
Downloaded: S_6a0bbad57cec83f864bfbeda.csv
Downloaded: S_6a982f42c0b872fe5acf3b4d.csv
Downloaded: S_6a98934d473d83d0fab3bae7.csv
Downloaded: S_6a989bd720a255be4ad69ac8.csv
Downloaded: S_6ab994c468c5614e19114e99.csv
All CSV fil

## Processing questionnaires and demographic data

In [12]:
from pathlib import Path
import re
import pandas as pd
from IPython.display import display

# ============================================================
# 1. Paths
# ============================================================

RAW_DIR = Path(
    r"C:\Users\zyfxl\OneDrive - UC Irvine\UCI\CCNL Lab\Foraging Task Overview"
    r"\foraging_MQLP-main\foraging_MQLP-main\replication sample\raw data\batch3"
)

OUT_FILE = RAW_DIR.parent / "replication_questionnaires_b3.csv"

# ============================================================
# 2. Scoring definitions
# ============================================================

QUIC_GROUPS = {
    "Parental_monitoring_and_involvement": [1, 3, 4, 5, 6, 7, 9, 10, 14],
    "Parental_predictability": [2, 8, 11, 12, 15, 16, 17, 31, 32, 33, 34, 35],
    "Parental_environment": [18, 19, 21, 22, 28, 29, 30],
    "Physical_environment": [13, 20, 26, 27, 36, 37, 38],
    "Safety_and_security": [23, 24, 25],

    "FA_factor_1": [2, 11, 12, 13, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30, 37, 38],
    "FA_factor_2": [1, 3, 4, 5, 6, 7, 8, 9, 10, 14, 15, 28],
    "FA_factor_3": [11, 31, 32, 33, 34, 35],
    "FA_factor_4": [20, 26, 27, 29],
    "FA_factor_5": [10, 17],
    "FA_factor_6": [36, 37, 38],
}

PCL_GROUPS = {
    "PCL_reexp": range(1, 6),
    "PCL_avoid": [6, 7],
    "PCL_neg_cog_mood": range(8, 15),
    "PCL_hypervi": range(15, 21),
}

PCL_CHOICES = {
    "not at all": 0,
    "a little bit": 1,
    "moderately": 2,
    "quite a bit": 3,
    "extremely": 4,
}

LEC_CHOICES = {
    "happened to me", "witnessed it", "learned about it",
    "part of my job", "not sure", "doesn't apply"
}

LEC_CRITERIA = {
    "LEC_Conservative": {
        "happened to me", "witnessed it"
    },
    "LEC_Stringent": {
        "happened to me", "witnessed it",
        "learned about it", "part of my job"
    },
    "LEC_Liberal": {
        "happened to me", "witnessed it",
        "learned about it", "part of my job", "not sure"
    },
}

# ============================================================
# 3. Parse raw questionnaire responses
# ============================================================

KEY = re.compile(r"(?:^|,)([A-Za-z_]\w*):")

def parse_response(line):
    parts = line.rstrip("\r\n").split(",", 3)

    if len(parts) < 4:
        return "", {}

    event = parts[0].strip().strip('"')
    raw = parts[3].lstrip()

    if not raw.startswith("{") or "}" not in raw:
        return event, {}

    obj = raw.split("}", 1)[0][1:]
    matches = list(KEY.finditer(obj))

    fields = {
        m.group(1): obj[
            m.end():matches[i + 1].start()
            if i + 1 < len(matches) else len(obj)
        ].strip()
        for i, m in enumerate(matches)
    }

    return event, fields


def total(values, complete=False):
    valid = [x for x in values if x is not None]

    if not valid or (complete and len(valid) != len(values)):
        return None

    return sum(valid)


# ============================================================
# 4. Process every subject
# ============================================================

rows = []
errors = []

for file in sorted(RAW_DIR.glob("*.csv")):

    rec = {
        "fileID": file.stem.removeprefix("S_"),
        "Age": None,
        "Gender": None,
    }

    quic, lec, pcl = {}, {}, {}

    try:
        with file.open(encoding="utf-8-sig", errors="replace") as f:

            for line in f:
                event, answers = parse_response(line)

                if event == "survey-html-form":
                    rec["fileID"] = answers.get(
                        "worker_id", rec["fileID"]
                    )

                elif event == "demo":
                    rec["Age"] = pd.to_numeric(
                        answers.get("age"), errors="coerce"
                    )
                    rec["Gender"] = answers.get("gender")

                elif event in {
                    "quic_prior_12", "quic_prior_18",
                    "lec_survey", "pcl_survey"
                }:

                    destination = (
                        quic if event.startswith("quic_")
                        else lec if event == "lec_survey"
                        else pcl
                    )

                    offset = 10 if event == "quic_prior_18" else 1

                    destination.update({
                        int(k[1:]) + offset: v
                        for k, v in answers.items()
                        if re.fullmatch(r"Q\d+", k)
                    })

        if not (quic or lec or pcl):
            continue

        # ====================================================
        # 5. QUIC: 38 binary questions
        # ====================================================

        for i in range(1, 39):
            rec[f"Q{i}"] = {
                "yes": 1,
                "no": 0
            }.get(quic.get(i, "").strip().lower())

        rec["QUIC"] = total([
            rec[f"Q{i}"] for i in range(1, 39)
        ])

        for name, items in QUIC_GROUPS.items():
            rec[name] = total([
                rec[f"Q{i}"] for i in items
            ])

        # ====================================================
        # 6. LEC: 17 event questions
        # ====================================================

        lec_scores = {
            name: [] for name in LEC_CRITERIA
        }

        for i in range(1, 18):

            raw = lec.get(i, "").strip().strip("[]")

            choices = {
                x.strip().lower().replace("’", "'")
                for x in raw.split(",") if x.strip()
            }

            valid = bool(choices) and choices.issubset(LEC_CHOICES)

            # Original direct-exposure score: 0, 1, or 2
            rec[f"LEC_Q{i}"] = (
                int("happened to me" in choices)
                + int("witnessed it" in choices)
                if valid else None
            )

            # Three cumulative exposure definitions
            # Each event counts at most once per definition.
            for name, qualifying in LEC_CRITERIA.items():

                score = (
                    int(bool(choices & qualifying))
                    if valid else None
                )

                lec_scores[name].append(score)

        # Original LEC total: 0-34
        rec["LEC_Total"] = total(
            [rec[f"LEC_Q{i}"] for i in range(1, 18)],
            complete=True
        )

        # Conservative / Stringent / Liberal: 0-17 each
        for name, scores in lec_scores.items():
            rec[name] = total(scores, complete=True)

        # ====================================================
        # 7. PCL-5: 20 questions, each scored 0-4
        # ====================================================

        for i in range(1, 21):
            rec[f"PCL_Q{i}"] = PCL_CHOICES.get(
                pcl.get(i, "").strip().lower()
            )

        rec["PCL_Total"] = total(
            [rec[f"PCL_Q{i}"] for i in range(1, 21)],
            complete=True
        )

        for name, items in PCL_GROUPS.items():
            rec[name] = total(
                [rec[f"PCL_Q{i}"] for i in items],
                complete=True
            )

        # ====================================================
        # 8. Questionnaire completion
        # ====================================================

        for survey, prefix, n in [
            ("QUIC", "Q", 38),
            ("LEC", "LEC_Q", 17),
            ("PCL", "PCL_Q", 20)
        ]:

            rec[f"{survey}_n_answered"] = sum(
                rec[f"{prefix}{i}"] is not None
                for i in range(1, n + 1)
            )

        rows.append(rec)

    except Exception as e:
        errors.append((file.name, str(e)))


# ============================================================
# 9. Combine subjects and save
# ============================================================

if not rows:
    raise ValueError("No questionnaire data found.")

df = pd.DataFrame(rows)

# Keep most complete record if subject has duplicate files
df["n_items"] = df[
    ["QUIC_n_answered", "LEC_n_answered", "PCL_n_answered"]
].sum(axis=1)

df = (
    df.sort_values("n_items", ascending=False)
      .drop_duplicates("fileID")
      .drop(columns="n_items")
      .sort_values("fileID")
      .reset_index(drop=True)
)

df.to_csv(OUT_FILE, index=False, encoding="utf-8-sig")

# ============================================================
# 10. Sanity check
# ============================================================

print(f"Total subjects: {len(df)}")
print(f"Complete QUIC: {(df.QUIC_n_answered == 38).sum()}")
print(f"Complete LEC: {(df.LEC_n_answered == 17).sum()}")
print(f"Complete PCL: {(df.PCL_n_answered == 20).sum()}")
print(f"Processing errors: {len(errors)}")
print(f"\nSaved to: {OUT_FILE}")

display(df[[
    "fileID", "Age", "Gender",
    "QUIC", "LEC_Total",
    "LEC_Conservative", "LEC_Stringent", "LEC_Liberal",
    "PCL_Total"
]].head())

Total subjects: 172
Complete QUIC: 156
Complete LEC: 172
Complete PCL: 172
Processing errors: 0

Saved to: C:\Users\zyfxl\OneDrive - UC Irvine\UCI\CCNL Lab\Foraging Task Overview\foraging_MQLP-main\foraging_MQLP-main\replication sample\raw data\replication_questionnaires_b3.csv


,fileID,Age,Gender,QUIC,LEC_Total,LEC_Conservative,LEC_Stringent,LEC_Liberal,PCL_Total
0,56e32190e414dc000c8b40ae,35,other,15,3,2,7,8,2
1,59497c498fb2140001afda85,33,female,14,0,0,0,0,4
2,59d9d9cd8fd07a0001842484,32,male,17,5,4,6,7,2
3,5a285307df86d80001455de5,30,male,17,9,6,14,14,10
4,5a3b209b2f3c2500018bc67d,27,male,16,0,0,5,5,8


## Add PRT relative to MVT 

In [ ]:
import pandas as pd
from pathlib import Path

data_dir = Path(
    r"C:\Users\zyfxl\OneDrive - UC Irvine\UCI\CCNL Lab\Foraging Task Overview"
    r"\foraging_MQLP-main\foraging_MQLP-main\replication sample\processed data\batch3"
)

avg = pd.read_csv(data_dir / "new_pilot_avg_batch3.csv")
df = pd.read_csv(data_dir / "replication_questionnaires_b3.csv")

# Standardize fileID in both datasets
for data in [avg, df]:
    data["fileID"] = (
        data["fileID"].astype("string")
        .str.strip()
        .str.removesuffix(".csv")
        .str.removeprefix("S_")
    )

# Galaxy: 0 = Poor, 1 = Neutral, 2 = Rich
planet_types = {
    0: "PRT_MVT_Poor",
    1: "PRT_MVT_Neutral",
    2: "PRT_MVT_Rich"
}

# Mean overharvesting by participant and planet type
prt = (
    avg.groupby(["fileID", "galaxy"])["prt_rel_om"]
    .mean()
    .unstack()
    .rename(columns=planet_types)
    .reset_index()
)

cols = list(planet_types.values())

# Equal-weighted average across planet types
prt["PRT_MVT_Overall"] = prt[cols].mean(axis=1, skipna=False)

# Merge
assert df["fileID"].is_unique
assert prt["fileID"].is_unique

new_cols = cols + ["PRT_MVT_Overall"]

df = df.drop(columns=new_cols, errors="ignore")
df = df.merge(prt, on="fileID", how="left", validate="one_to_one", indicator=True)

# Diagnostic
print(df["_merge"].value_counts())
print("Questionnaire participants:", len(df))
print("Participants matched:", (df["_merge"] == "both").sum())
print("Missing overall PRT:", df["PRT_MVT_Overall"].isna().sum())

display(df.loc[df["PRT_MVT_Overall"].isna(),
               ["fileID", "_merge"] + new_cols])

# Save separately
df = df.drop(columns="_merge")
df.to_csv(data_dir / "replication_questionnaires.csv", index=False)

display(df[["fileID"] + new_cols].head())

_merge
both          170
left_only       2
right_only      0
Name: count, dtype: int64
Questionnaire participants: 172
Participants matched: 170
Missing overall PRT: 3


,fileID,_merge,PRT_MVT_Poor,PRT_MVT_Neutral,PRT_MVT_Rich,PRT_MVT_Overall
95,697938c09291ba65dedf7b64,both,48.0,70.0,NaN,NaN
109,69a9cac25a062cd320e83a1,left_only,NaN,NaN,NaN,NaN
130,69fb7a05cbb59761b9e31492,left_only,NaN,NaN,NaN,NaN


,fileID,PRT_MVT_Poor,PRT_MVT_Neutral,PRT_MVT_Rich,PRT_MVT_Overall
0,56e32190e414dc000c8b40ae,0.833333,1.040000,1.578947,1.150760
1,59497c498fb2140001afda85,0.312500,0.576923,-1.083333,-0.064637
2,59d9d9cd8fd07a0001842484,0.384615,1.040000,1.947368,1.123995
3,5a285307df86d80001455de5,0.666667,0.692308,2.166667,1.175214
4,5a3b209b2f3c2500018bc67d,0.000000,0.615385,0.545455,0.386946
